In [1]:
# Install the core engine, web server, and tunnel tools
!pip install llama-cpp-python fastapi uvicorn pyngrok nest_asyncio

# Download the model file (TinyLlama)
!wget -q https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf -O model.gguf

print("✅ Foundation ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 MB 16.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.20-py3-none-linux_x86_64.whl size=13152929 sha256=e779124e74559c14c7e07746577347ae51ac270e0b5ccebd4b24b8a9eca19a3b
  Stored in directory: /root/.cache/pip/wheels/54/af/76/8c15ef256bcc7c70e0b033c10929b08aae811ef1eac47c6764
Successfully built llama-cpp-python
✅ Foundation ready.


In [2]:
from llama_cpp import Llama

print("🧠 Loading AI Model into Virtual RAM... this may take a moment.")
# We define 'llm' here so the API can find it later
llm = Llama(model_path="./model.gguf", n_ctx=512)
print("✅ Model loaded and ready.")

llama_model_loader: loaded meta data with 23 key-value pairs and 201 tensors from ./model.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = tinyllama_tinyllama-1.1b-chat-v1.0
llama_model_loader: - kv   2:                       llama.context_length u32              = 2048
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 5632
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 64
llama_model_loader: - kv   7:                 llama.attention.head_count u

🧠 Loading AI Model into Virtual RAM... this may take a moment.


print_info: arch                  = llama
print_info: vocab_only            = 0
print_info: no_alloc              = 0
print_info: n_ctx_train           = 2048
print_info: n_embd                = 2048
print_info: n_embd_inp            = 2048
print_info: n_layer               = 22
print_info: n_head                = 32
print_info: n_head_kv             = 4
print_info: n_rot                 = 64
print_info: n_swa                 = 0
print_info: is_swa_any            = 0
print_info: n_embd_head_k         = 64
print_info: n_embd_head_v         = 64
print_info: n_gqa                 = 8
print_info: n_embd_k_gqa          = 256
print_info: n_embd_v_gqa          = 256
print_info: f_norm_eps            = 0.0e+00
print_info: f_norm_rms_eps        = 1.0e-05
print_info: f_clamp_kqv           = 0.0e+00
print_info: f_max_alibi_bias      = 0.0e+00
print_info: f_logit_scale         = 0.0e+00
print_info: f_attn_scale          = 0.0e+00
print_info: n_ff                  = 5632
print_info: n_expert       

✅ Model loaded and ready.


Using gguf chat template: {% for message in messages %}
{% if message['role'] == 'user' %}
{{ '<|user|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'system' %}
{{ '<|system|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'assistant' %}
{{ '<|assistant|>
'  + message['content'] + eos_token }}
{% endif %}
{% if loop.last and add_generation_prompt %}
{{ '<|assistant|>' }}
{% endif %}
{% endfor %}
Using chat eos_token: </s>
Using chat bos_token: <s>


In [3]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class PromptRequest(BaseModel):
    prompt: str

@app.post("/generate")
async def generate(request: PromptRequest):
    # This connects the incoming request to the 'llm' brain we loaded in Cell 2
    output = llm(
        f"<|system|>\nYou are a helpful assistant.</s>\n<|user|>\n{request.prompt}</s>\n<|assistant|>\n",
        max_tokens=200,
        stop=["</s>"],
        echo=False
    )
    return {
        "status": "success",
        "worker_id": "Colab-Worker-1",
        "response": output["choices"][0]["text"]
    }
print("✅ API routes defined.")

✅ API routes defined.


In [4]:
@app.get("/health")
async def health_check():
    # This is like the worker saying "I am awake and my brain is loaded!"
    return {"status": "healthy", "worker_id": "Colab-Worker-1"} # Change to Worker-2 for the other tab

In [5]:
from pyngrok import ngrok

# 1. PASTE YOUR FIRST ACCOUNT TOKEN HERE
NGROK_TOKEN = "3CCPdr9GIwUQQB81Yw5nS2PDp4C_7S8CdRFu7nFpM6aGaqWzS"

# 2. Authenticate and open the bridge
ngrok.kill() # Closes any old "ghost" tunnels
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000)

print(f"\n🚀 WORKER 1 IS LIVE AT: {public_url}")
print("👉 Copy the URL above and put it in your Master Script.")


🚀 WORKER 1 IS LIVE AT: NgrokTunnel: "https://wincing-alumni-lark.ngrok-free.dev" -> "http://localhost:8000"
👉 Copy the URL above and put it in your Master Script.


In [6]:
import threading
import nest_asyncio
import uvicorn

nest_asyncio.apply()

def run_worker():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

# Start the background thread
threading.Thread(target=run_worker, daemon=True).start()
print("✅ Server thread is running in the background.")

✅ Server thread is running in the background.


In [7]:
import requests

# This automatically uses the URL generated in Cell 4
test_url = public_url.public_url + "/generate"
payload = {"prompt": "Who is trump?"}

print(f"Testing Worker 1 at {test_url}...")
try:
    response = requests.post(test_url, json=payload, timeout=60)
    print("\n--- Worker 1 Response ---")
    print(response.json())
except Exception as e:
    print(f"❌ Connection Error: {e}")

Testing Worker 1 at https://wincing-alumni-lark.ngrok-free.dev/generate...


llama_perf_context_print:        load time =    5819.10 ms
llama_perf_context_print: prompt eval time =    5818.77 ms /    37 tokens (  157.26 ms per token,     6.36 tokens per second)
llama_perf_context_print:        eval time =   23502.65 ms /   199 runs   (  118.10 ms per token,     8.47 tokens per second)
llama_perf_context_print:       total time =   29434.56 ms /   236 tokens
llama_perf_context_print:    graphs reused =        198



--- Worker 1 Response ---
{'status': 'success', 'worker_id': 'Colab-Worker-1', 'response': "Trump is an American businessman, politician, and television personality. He is the 45th President of the United States of America.\n\nTrump was born in New York City in 1946 and grew up in Queens, New York. He attended the New York Military Academy for his early education, where he was an outstanding athlete. After graduation, he served as a combat infantryman in the United States Marine Corps, where he earned several awards and decorations.\n\nTrump's political career began in the early 1980s as a real estate developer in New York City. He later became a successful businessman, launching several successful enterprises, including the Trump Plaza Hotel and Casino in Atlantic City.\n\nTrump's political career gained momentum in the 1980s when he made his first political bid for the U.S. House of Representatives. He won the election"}
